# Checagem reproduzível — módulo 52

## tl;dr

- A fonte e o snapshot da rodada têm o mesmo SHA-256 e contêm somente o módulo 52.
- O CP-SAT provou o ótimo: 363 de 402 transmissões alocadas (90,30%), com 196 docentes utilizados.
- As 39 pendências se dividem em 17 choques de horário, 11 ausências de perfil/carga, 9 combinações de capacidade e horário, 1 agenda inválida e 1 capacidade esgotada.
- A auditoria foi aprovada sem ocorrências; não há colisões de horário nem violações da política de capacidade.
- A base foi aprovada com ressalvas: 2 grupos altos e 2 médios, nenhum bloqueante.


## Context & Methods

Este notebook reconcilia o manifesto, os relatórios JSON e a planilha final da `rodada_002`. Ele não recalcula a otimização; verifica a integridade e a coerência dos artefatos publicados.

### Key Assumptions

- `OPTIMAL` representa o ótimo dentro das regras atualmente implementadas.
- Cada transmissão consome 2 horas.
- Docentes `CLT STRICTO` podem receber no máximo uma transmissão, mesmo com `CH_LETIVA=0`; os demais respeitam `CH_ALOCADA <= CH_LETIVA`.
- Linhas Sinérgicas são informativas e ficam fora do conjunto alocável.


In [1]:
from collections import Counter, defaultdict
from hashlib import sha256
import json
from pathlib import Path

from openpyxl import load_workbook

ROUND_DIR = Path.cwd().parent.resolve()
SOURCE_NAME = 'BASE_SINTETICA_PERFIL_DOCENTE_COMPLETO.xlsx'
SOURCE_ORIGINAL = ROUND_DIR.parents[1] / SOURCE_NAME
SOURCE_SNAPSHOT = ROUND_DIR / 'fonte' / SOURCE_NAME
RESULT_WORKBOOK = ROUND_DIR / 'alocacao' / 'resultado_alocacao.xlsx'

manifest = json.loads((ROUND_DIR / 'manifesto.json').read_text(encoding='utf-8'))
validation = json.loads((ROUND_DIR / 'validacao' / 'relatorio_validacao.json').read_text(encoding='utf-8'))
summary = json.loads((ROUND_DIR / 'alocacao' / 'resumo_alocacao.json').read_text(encoding='utf-8'))
audit = json.loads((ROUND_DIR / 'auditoria' / 'auditoria_alocacao.json').read_text(encoding='utf-8'))

ROUND_DIR


WindowsPath('C:/Users/48960380881/Documents/GitHub/FerramentaAlocação/TESTEM52/resultado/rodada_002')

## Data

### 1. Verify source identity and module


In [2]:
source_hashes = {
    'original': sha256(SOURCE_ORIGINAL.read_bytes()).hexdigest(),
    'snapshot': sha256(SOURCE_SNAPSHOT.read_bytes()).hexdigest(),
    'manifest': manifest['source']['sha256'],
}
source_check = {
    'sha_match': len(set(source_hashes.values())) == 1,
    'sha256': source_hashes['original'],
    'modules': validation['metadata']['modules'],
    'expected_module': validation['metadata']['expected_module'],
    'map_rows': validation['sheets']['MAPA PEDAGÓGICO']['rows'],
    'teacher_rows': validation['sheets']['DOCENTES']['rows'],
    'allocating_rows': validation['metadata']['allocating_rows'],
}
assert source_check['sha_match']
assert source_check['modules'] == [52]
assert source_check['expected_module'] == 52
source_check


{'sha_match': True,
 'sha256': '5dbc22c481c36d60e7747f01d48337c3f6752e097519a0dcf77ce6892dfb9e76',
 'modules': [52],
 'expected_module': 52,
 'map_rows': 990,
 'teacher_rows': 233,
 'allocating_rows': 402}

## Results

### 2. Reconcile counts and hard constraints


In [3]:
workbook = load_workbook(RESULT_WORKBOOK, read_only=True, data_only=True)
allocation_sheet = workbook['ALOCACOES']
allocation_values = list(allocation_sheet.iter_rows(values_only=True))
allocation_headers = allocation_values[0]
allocations = [dict(zip(allocation_headers, row)) for row in allocation_values[1:]]

teacher_sheet = workbook['DOCENTES']
teacher_values = list(teacher_sheet.iter_rows(values_only=True))
teacher_headers = teacher_values[0]
teachers = [dict(zip(teacher_headers, row)) for row in teacher_values[1:]]
workbook.close()

status_key, reason_key, source_row_key = allocation_headers[0], allocation_headers[1], allocation_headers[3]
day_key, time_key = allocation_headers[11], allocation_headers[12]
teacher_name_key, badge_key, candidate_key = allocation_headers[18], allocation_headers[19], allocation_headers[20]
teacher_badge_key, capacity_key, teacher_status_key = teacher_headers[1], teacher_headers[4], teacher_headers[6]
allocated_hours_key, transmission_count_key, contract_key = teacher_headers[8], teacher_headers[11], teacher_headers[13]

status_counts = Counter(row[status_key] for row in allocations)
allocated = [row for row in allocations if row[status_key] == 'ALOCADA']
unassigned = [row for row in allocations if row[status_key] == 'NAO_ALOCADA']
used_badges = {row[badge_key] for row in allocated}
active_teachers = [row for row in teachers if str(row[teacher_status_key]).upper() == 'ATIVO']

slots = defaultdict(list)
for row in allocated:
    slots[(row[badge_key], row[day_key], row[time_key])].append(row[source_row_key])
slot_conflicts = {slot: rows for slot, rows in slots.items() if len(rows) > 1}

capacity_violations = []
stricto_violations = []
for teacher in teachers:
    contract = str(teacher[contract_key] or '').upper()
    allocated_hours = teacher[allocated_hours_key] or 0
    if contract == 'CLT STRICTO':
        if (teacher[transmission_count_key] or 0) > 1:
            stricto_violations.append(teacher[teacher_badge_key])
    elif allocated_hours > (teacher[capacity_key] or 0):
        capacity_violations.append(teacher[teacher_badge_key])

result_check = {
    'transmissions': len(allocations),
    'allocated': len(allocated),
    'unassigned': len(unassigned),
    'coverage_pct': round(100 * len(allocated) / len(allocations), 2),
    'allocated_hours': sum((teacher[allocated_hours_key] or 0) for teacher in teachers),
    'active_teachers': len(active_teachers),
    'used_teachers': len(used_badges),
    'active_teacher_coverage_pct': round(100 * len(used_badges) / len(active_teachers), 2),
    'unique_source_rows': len({row[source_row_key] for row in allocations}),
    'slot_conflicts': slot_conflicts,
    'capacity_violations': capacity_violations,
    'stricto_violations': stricto_violations,
    'audit_status': audit['status'],
    'audit_issue_count': audit['issue_count'],
}
assert result_check['transmissions'] == summary['transmissions'] == audit['checks']['transmissions']
assert result_check['allocated'] == summary['allocated'] == audit['checks']['allocated']
assert result_check['unassigned'] == summary['unassigned'] == audit['checks']['unassigned']
assert result_check['used_teachers'] == summary['used_teachers'] == audit['checks']['used_teachers']
assert result_check['allocated_hours'] == summary['allocated_hours']
assert result_check['unique_source_rows'] == result_check['transmissions']
assert not slot_conflicts and not capacity_violations and not stricto_violations
assert audit['status'] == 'APROVADO' and audit['issue_count'] == 0
result_check


{'transmissions': 402,
 'allocated': 363,
 'unassigned': 39,
 'coverage_pct': 90.3,
 'allocated_hours': 726,
 'active_teachers': 229,
 'used_teachers': 196,
 'active_teacher_coverage_pct': 85.59,
 'unique_source_rows': 402,
 'slot_conflicts': {},
 'capacity_violations': [],
 'stricto_violations': [],
 'audit_status': 'APROVADO',
 'audit_issue_count': 0}

### 3. Explain unassigned transmissions


In [4]:
reason_counts = Counter(row[reason_key] for row in unassigned)
reason_table = [
    {
        'reason': reason,
        'count': count,
        'share_of_unassigned_pct': round(100 * count / len(unassigned), 2),
        'source_rows': [row[source_row_key] for row in unassigned if row[reason_key] == reason],
    }
    for reason, count in reason_counts.most_common()
]
assert dict(reason_counts) == summary['unassigned_reasons']
assert sum(item['count'] for item in reason_table) == len(unassigned)
reason_table


[{'reason': 'CHOQUE_DE_HORARIO',
  'count': 17,
  'share_of_unassigned_pct': 43.59,
  'source_rows': [47,
   74,
   88,
   173,
   276,
   279,
   309,
   332,
   393,
   403,
   505,
   531,
   811,
   877,
   878,
   880,
   894]},
 {'reason': 'SEM_DOCENTE_COM_PERFIL_E_CARGA',
  'count': 11,
  'share_of_unassigned_pct': 28.21,
  'source_rows': [27, 139, 141, 142, 265, 369, 518, 720, 776, 883, 884]},
 {'reason': 'CAPACIDADE_E_HORARIO_COMBINADOS',
  'count': 9,
  'share_of_unassigned_pct': 23.08,
  'source_rows': [26, 28, 92, 97, 124, 401, 482, 690, 752]},
 {'reason': 'AGENDA_INVALIDA',
  'count': 1,
  'share_of_unassigned_pct': 2.56,
  'source_rows': [543]},
 {'reason': 'CAPACIDADE_LETIVA_ESGOTADA',
  'count': 1,
  'share_of_unassigned_pct': 2.56,
  'source_rows': [762]}]

### 4. Map validation caveats to the result


In [5]:
validation_findings = [
    {
        'severity': issue['severity'],
        'code': issue['code'],
        'count': issue['count'],
        'rows': issue['rows'],
        'blocking': issue['blocking'],
    }
    for issue in validation['issues']
]

no_profile_issue = next(issue for issue in validation['issues'] if issue['code'] == 'PERFIL_SEM_DOCENTE_ATIVO')
no_profile_result_rows = {row[source_row_key] for row in unassigned if row[reason_key] == 'SEM_DOCENTE_COM_PERFIL_E_CARGA'}
assert set(no_profile_issue['rows']) == no_profile_result_rows

invalid_schedule_rows = {row[source_row_key] for row in unassigned if row[reason_key] == 'AGENDA_INVALIDA'}
assert invalid_schedule_rows == {543}

divergence_issue = next(issue for issue in validation['issues'] if issue['code'] == 'ORDEM_METODOLOGIA_DIVERGENTE')
divergent_rows = set(divergence_issue['rows'])
divergent_allocating = [row for row in allocations if row[source_row_key] in divergent_rows]
divergence_outcomes = Counter((row[status_key], row[reason_key] or '') for row in divergent_allocating)

quality_impact = {
    'validation_status': validation['status'],
    'severity_counts': validation['summary']['by_severity'],
    'blocking_issue_groups': validation['summary']['blocking_issue_groups'],
    'findings': validation_findings,
    'profile_warning_matches_result': set(no_profile_issue['rows']) == no_profile_result_rows,
    'missing_schedule_matches_result': sorted(invalid_schedule_rows),
    'divergent_rows_total': len(divergent_rows),
    'divergent_rows_in_allocation_scope': len(divergent_allocating),
    'divergence_outcomes': dict(divergence_outcomes),
}
quality_impact


{'validation_status': 'APROVADO_COM_RESSALVAS',
 'severity_counts': {'CRÍTICA': 0, 'ALTA': 2, 'MÉDIA': 2, 'BAIXA': 0},
 'blocking_issue_groups': 0,
 'findings': [{'severity': 'MÉDIA',
   'code': 'CAMPO_OBRIGATORIO_VAZIO',
   'count': 1,
   'rows': [546],
   'blocking': False},
  {'severity': 'ALTA',
   'code': 'OFERTA_SEM_HORARIO',
   'count': 1,
   'rows': [543],
   'blocking': False},
  {'severity': 'MÉDIA',
   'code': 'ORDEM_METODOLOGIA_DIVERGENTE',
   'count': 19,
   'rows': [28,
    81,
    90,
    123,
    142,
    173,
    270,
    295,
    402,
    505,
    531,
    543,
    665,
    762,
    813,
    843,
    878,
    908,
    924],
   'blocking': False},
  {'severity': 'ALTA',
   'code': 'PERFIL_SEM_DOCENTE_ATIVO',
   'count': 11,
   'rows': [27, 139, 141, 142, 265, 369, 518, 720, 776, 883, 884],
   'blocking': False}],
 'profile_warning_matches_result': True,
 'missing_schedule_matches_result': [543],
 'divergent_rows_total': 19,
 'divergent_rows_in_allocation_scope': 16,
 '

## Takeaways

1. A solução é confiável dentro das regras atuais: todas as fases foram `OPTIMAL`, o snapshot é idêntico à fonte e a auditoria não encontrou violações.
2. Das 39 pendências, 27 ainda têm candidato e são limitadas por horário/capacidade; este é o principal espaço para simulações de remanejamento.
3. As 11 pendências sem perfil compatível exigem ampliação/correção do cadastro docente, não mais tempo de solver.
4. A linha 543 precisa de horário para voltar ao modelo; a linha 546 sem currículo é Sinérgica e não afeta as 402 ofertas processadas.
5. As 19 divergências entre ORDEM e METODOLOGIA, sendo 16 no escopo alocável, devem ser confirmadas antes de transformar esta rodada em decisão definitiva.
